To copy this template: File -> Save a Copy in Drive

***DISCLAIMER**: In case of any discrepancy in the assignment instruction, please refer to the `PDF` document.*

# Problem 2 - Using BERT for question answering  **(10)**

In this question, we will use a pre-trained model for generating answers to a question based on a paragraph.

In [ ]:
# Install the transformers library that will be used for BERT models.
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers'])

## 2.1 **(1)**

We will use the BertForQuestionAnswering model and the BertTokenizer as our tokenizer.

In [ ]:
import torch
from transformers import BertForQuestionAnswering
from transformers import BertTokenizer

#Get the pretrained 'bert-large-uncased-whole-word-masking-finetuned-squad' model from the BertForQuestionAnswering library
model = BertForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')


# Get the matching tokenizer from the same pretrained checkpoint.
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')


We define the question as well as the textual paragraph which the question is based on.


In [ ]:
question = "What was BERT trained on?"

paragraph = "BERT stands for Bidirectional Encoder Representation of Transformer. I feel that its name itself is descriptive enough to get the gist. Still, to understand it better, it’s encoder part of the encoder-decoder transformer model, it’s also bidirectional in nature, which means that for any input it’s able to learn dependencies from both left and right of any word. It was trained on Wikipedia text and BooksCorpus and open-sourced back in 2018 by Google. You can find the official repository and paper at Github: BERT and BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. There are two models introduced in the paper. BERT base — 12 layers (transformer blocks), 110 million parameters. BERT Large — 24 layers, 340 million parameters. Later google also released Multi-lingual BERT to accelerate the research"

## 2.2 **(2)**

Use the encode_plus function. Define the text parameter as the question, and the text_pair as the paragraph.

You can refer to: https://huggingface.co/docs/transformers/v4.19.0/en/main_classes/tokenizer#transformers.PreTrainedTokenizer.__call__

In [ ]:
encoding = tokenizer(text=question, text_pair=paragraph, add_special_tokens=True)

## 2.3 **(2)**

The encoding is a dictionary with multiple keys. Your task is to identify which keys will be used for the inputs and which will be used for the segment embeddings.

In [ ]:
print(encoding.keys())

In [ ]:
# input_ids are the token ids and token_type_ids are the segment embeddings.
inputs = encoding['input_ids']  # Token embeddings

sentence_embedding = encoding['token_type_ids']  # Segment embeddings


# we convert the input ids to tokens
tokens = tokenizer.convert_ids_to_tokens(inputs) #input tokens

The model returns the most probable start and end words scores.

In [ ]:
scores = model(input_ids=torch.tensor([inputs]), token_type_ids=torch.tensor([sentence_embedding]))
print(scores)

## 2.4 **(2)**

Now we have start scores and end scores we can get both the start index and the end index and use both the indices for span prediction.

In [ ]:
start_index = torch.argmax(scores.start_logits).item()

end_index = torch.argmax(scores.end_logits).item()


if end_index >= start_index:
    get = " ".join(tokens[start_index:end_index+1])
    cleaned_answer = tokenizer.convert_tokens_to_string(tokens[start_index:end_index+1])
else:
    print("I am unable to find the answer to this question. Can you please ask another question?")

## 2.5 **(1)**
Display the answer given by the model.

In [ ]:
print('Raw token answer:', get)
print('Cleaned answer:', cleaned_answer)

## 2.6 **(2)**

Did you see any unusual tokens in the answer? What could be the reason for that?

**Answer**: Yes. The raw span can contain unusual tokens such as `wiki ##pedia` or `books ##corpus`. This happens because BERT uses a WordPiece tokenizer, which breaks some words into subword pieces when the whole word is not stored as a single token in the vocabulary. The cleaned answer merges those pieces back into normal text.